# Quiz 04: Web APIs and JSON

This quiz covers web APIs and JSON, from lectures 18 and 19.

- You have 75 minutes. The quiz is open-book and open-notes
- Fork and clone this repository, and work in your clone
- The data files you need are already in the `data/` folder
- Run the cells from top to bottom, and run them all again before you submit
- Tasks 4, 6, 7, 8, 9 and 10 send real requests to live APIs
- Tasks 7 and 10 run `curl` in the terminal. Paste the commands and their output into the answer cells

Write your answers in the empty cells below each task. The bonus task is optional.


In [31]:
import json

import requests
import pandas as pd

## Task 1

Look at this URL:

```
https://api.worldbank.org/v2/country/URY/indicator/SP.POP.TOTL?format=json
```

Write the scheme, the host, the path and the query string, one on each line. Then write the full URL for France's population from 2015 to 2018.

**Your answer:**

Scheme: https

Host: api.worldbank.org

Path: vs/country/URY/indicator/SP.POP.TOTL

Query string: format=json

URL: https://api.worldbank.org/v2/country/FRA/indicator/SP.POP.TOTL?format=json&date=2015:2018

## Task 2

Load `data/shuttle.json` with `json.load` into a variable called `reply`. Print:

1. the name of the stop
2. how many arrivals the reply lists
3. the wait for the second arrival, with its unit, for example `5 minutes`


In [32]:
# Task 2

with open("data/shuttle.json") as f:
    reply = json.load(f)

print(json.dumps(reply, indent=2))

{
  "stop": {
    "id": "EM-12",
    "name": "Emory Point",
    "location": {
      "lat": 33.79,
      "lon": -84.32
    }
  },
  "updated": "2026-11-19T10:15",
  "units": {
    "wait": "minutes",
    "distance": "km"
  },
  "arrivals": [
    {
      "route": "C",
      "wait": 4,
      "distance": 1.2
    },
    {
      "route": "Cliff 6",
      "wait": 11,
      "distance": 3.5
    },
    {
      "route": "C",
      "wait": 19,
      "distance": 6.0
    }
  ]
}


In [33]:
print(reply["stop"]["name"])
print(len(reply["arrivals"]))
print(reply["arrivals"][1]["wait"], reply["units"]["wait"])

Emory Point
3
11 minutes


## Task 3

Load `data/wb_electricity_ken.json` into a variable called `wb`. It holds the share of Kenya's (KEN) population with access to electricity. Like many APIs, the World Bank sends a list with two elements: metadata first, then the records. Print the number of records the metadata says there are, and check it against the length of the records list. Then print the value for 2020.


In [34]:
# Task 3

with open("data/wb_electricity_ken.json") as f:
    wb = json.load(f)

print(json.dumps(wb, indent=2))

meta = wb[0]
records = wb[1]

print("Records in metadata:", meta["total"])
print("Length of records:", len(records))

for record in records:
    if record["date"] == "2020":
        print("2020 value:", record["value"])


[
  {
    "page": 1,
    "pages": 1,
    "per_page": 50,
    "total": 13,
    "sourceid": "2",
    "lastupdated": "2026-11-11"
  },
  [
    {
      "indicator": {
        "id": "EG.ELC.ACCS.ZS",
        "value": "Access to electricity (% of population)"
      },
      "country": {
        "id": "KE",
        "value": "Kenya"
      },
      "countryiso3code": "KEN",
      "date": "2024",
      "value": 77,
      "unit": "",
      "obs_status": "",
      "decimal": 1
    },
    {
      "indicator": {
        "id": "EG.ELC.ACCS.ZS",
        "value": "Access to electricity (% of population)"
      },
      "country": {
        "id": "KE",
        "value": "Kenya"
      },
      "countryiso3code": "KEN",
      "date": "2023",
      "value": 76.2,
      "unit": "",
      "obs_status": "",
      "decimal": 1
    },
    {
      "indicator": {
        "id": "EG.ELC.ACCS.ZS",
        "value": "Access to electricity (% of population)"
      },
      "country": {
        "id": "KE",
        "value

The metadata shows 13 records, which matches the actual length of the records list. Kenya's access to electricity in 2020 was 71.5% of the population.

## Task 4

Ask [Open-Meteo](https://open-meteo.com/en/docs) (`https://api.open-meteo.com/v1/forecast`) for the current wind speed at Emory: latitude 33.79, longitude -84.32, and `wind_speed_10m` as the `current` variable in the query. Pass the three values as a dict to `params`, with `timeout=10`, and save the reply as `r`. Print `r.url` and `r.status_code`. Then print the wind speed with its unit.


In [35]:
# Task 4

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 33.79,
    "longitude": -84.32,
    "current": "wind_speed_10m"
}

r = requests.get(url, params=params, timeout=10)

print(r.url)
print(r.status_code)

r.raise_for_status()

data = r.json()

wind_speed = data["current"]["wind_speed_10m"]
unit = data["current_units"]["wind_speed_10m"]

print(wind_speed, unit)


https://api.open-meteo.com/v1/forecast?latitude=33.79&longitude=-84.32&current=wind_speed_10m
200
8.4 km/h


## Task 5

Without running any code, write the full URL that this call sends. Some characters must be percent-encoded in a URL. Encode them as `requests` would (lecture 18).

```python
requests.get("https://api.example.com/v2/books",
             params={"author": "Toni Morrison", "year": "1970:1990", "format": "json"})
```


**Your answer:**

https://api.example.com/v2/books?author=Toni%20Morrison&year=1970%3A1990&format=json

## Task 6

Make two bad requests and print the status code and the body (`r.json()`) of each:

1. The Task 4 request with latitude 9999
2. `https://api.worldbank.org/v2/country/KEN/indicator/EG.ELC.ACCS.ZZ?format=json` (the indicator is misspelled)

For each, say in one sentence whether `raise_for_status()` would stop the script. Then write one line of code that would catch the second error.


In [36]:
# Task 6

# Request 1: Invalid latitude
url1 = "https://api.open-meteo.com/v1/forecast"

params1 = {
    "latitude": 9999,
    "longitude": -84.32,
    "current": "wind_speed_10m"
}

r1 = requests.get(url1, params=params1, timeout=10)

print("Request 1 status code:", r1.status_code)
print("Request 1 body:", r1.json())

# Request 2: Invalid World Bank indicator
url2 = (
    "https://api.worldbank.org/v2/country/KEN/"
    "indicator/EG.ELC.ACCS.ZZ"
)

r2 = requests.get(
    url2,
    params={"format": "json"},
    timeout=10
)

print("Request 2 status code:", r2.status_code)
print("Request 2 body:", r2.json())

Request 1 status code: 400
Request 1 body: {'reason': 'Latitude must be in range of -90 to 90°. Given: 9999.0.', 'error': True}
Request 2 status code: 200
Request 2 body: [{'message': [{'id': '120', 'key': 'Invalid value', 'value': 'The provided parameter value is not valid'}]}]


**Your answer:**

The first request returned status code 400 because the latitude was outside the valid range of -90 to 90 degrees. raise_for_status() would catch this error.

The second request returned status code 200, but the JSON body still contained an "Invalid value" message. raise_for_status() would not catch this because the HTTP status was 200. This shows why we need to check the actual response data instead of only checking the status code.

## Task 7

In the terminal, use `curl` to ask [Open-Meteo](https://open-meteo.com/en/docs) (`https://api.open-meteo.com/v1/forecast`) for the weather in Montevideo: latitude -34.90, longitude -56.19, and `relative_humidity_2m` as the `current` variable. Paste into the answer cell, as text:

1. The `curl` command that prints only the status line, and its output
2. The `curl` command that saves the reply as `montevideo.json`, in the top folder of the repository
3. The Python command that prints the saved file indented, and the humidity with its unit


**Your answer:**


In [37]:
%%bash

BASE="https://api.open-meteo.com/v1/forecast"
Q="latitude=-34.90&longitude=-56.19&current=relative_humidity_2m"

# 1. Print the HTTP status line
curl -si "$BASE?$Q" | head -n 1

# 2. Save the response
curl -s -o montevideo.json "$BASE?$Q"

# 3. Print the saved JSON
python3 -m json.tool montevideo.json

HTTP/1.1 200 OK
{
    "latitude": -34.90334,
    "longitude": -56.1929,
    "generationtime_ms": 0.0286102294921875,
    "utc_offset_seconds": 0,
    "timezone": "GMT",
    "timezone_abbreviation": "GMT",
    "elevation": 7.0,
    "current_units": {
        "time": "iso8601",
        "interval": "seconds",
        "relative_humidity_2m": "%"
    },
    "current": {
        "time": "2026-11-19T23:30",
        "interval": 900,
        "relative_humidity_2m": 82
    }
}


In [38]:
with open("montevideo.json") as f:
    data = json.load(f)

print(data["current"]["relative_humidity_2m"])
print(data["current_units"]["relative_humidity_2m"])

82
%


## Task 8

The script `scripts/pull_data.py` downloads one World Bank indicator and saves two files in `data/raw/`. Change its four settings to collect the share of the population using the internet in Kenya, Ghana and Senegal, from 2015 to 2024, with `internet` as the output name. Find the indicator code at [data.worldbank.org/indicator](https://data.worldbank.org/indicator).

Run the script from the top folder of this repository, and paste the settings you used and the script's full output into the answer cell. Then answer briefly:

1. How many rows did you get, and why does the script save the raw reply as well as the CSV?
2. Look at Ghana's values in `data/raw/internet.csv`. Would you trust the value for 2019? Say why, and what you would do about it in an analysis.


Settings changed:

INDICATOR = "IT.NET.USER.ZS"

COUNTRIES = "KEN;GHA;SEN"

YEARS = "2015:2024"

OUTPUT_NAME = "internet"

In [39]:
%%bash

python3 scripts/pull_data.py

Requesting https://api.worldbank.org/v2/country/KEN;GHA;SEN/indicator/IT.NET.USER.ZS
Saved 30 rows to data/raw/internet.csv
  country_code country  year      value
0          GHA   Ghana  2015  23.000000
1          GHA   Ghana  2016  28.000000
2          GHA   Ghana  2017  37.884201
3          GHA   Ghana  2018  43.000000
4          GHA   Ghana  2019  22.073200


In [40]:
%%bash

python3 -c 'import pandas as pd; df=pd.read_csv("data/raw/internet.csv"); print(df[(df["country"]=="Ghana") & (df["year"].between(2018, 2020))].sort_values("year").to_string(index=False)); print("Total rows:", len(df))'

country_code country  year     value
         GHA   Ghana  2018 43.000000
         GHA   Ghana  2019 22.073200
         GHA   Ghana  2020 62.517601
Total rows: 30


**Your answer:**

The script saved 30 rows in the CSV file. It saves both the raw JSON and the CSV because the JSON keeps the original API response, while the CSV is easier to use for analysis. Keeping the raw data also means we can check it again later without making another API request.

I would be careful with Ghana's 2019 value because internet usage dropped from 43% in 2018 to about 22% in 2019, then increased to about 63% in 2020. This seems like a very large change in just two years. Before using the data, I would check the original source to see if there was a reporting problem or a change in how the data was collected.

## Task 9

[Frankfurter](https://frankfurter.dev/v1/) is a free API for exchange rates. Its latest rates are at `https://api.frankfurter.dev/v1/latest`, and it takes two options: `base`, the currency you convert from, and `symbols`, the currencies you want, separated by commas.

Using `requests` and `params`, ask for the latest rates from `USD` to `BRL` and `EUR`. Print `r.url`, the date of the rates, and the BRL rate.

For a range of dates, replace `latest` with `start..end`, for example `2020-01-06..2020-01-19` (two weeks, Monday to Sunday). Request that range with the same `params`. Print how many days of rates you got and the BRL rate on 15 January 2020. In one sentence, say why there are fewer than 14 days.


In [41]:
# Task 9

url = "https://api.frankfurter.dev/v1/latest"

params = {
    "base": "USD",
    "symbols": "BRL,EUR"
}

r = requests.get(url, params=params, timeout=10)
r.raise_for_status()

data = r.json()

print("URL:", r.url)
print("Date:", data["date"])
print("BRL rate:", data["rates"]["BRL"])

URL: https://api.frankfurter.dev/v1/latest?base=USD&symbols=BRL%2CEUR
Date: 2026-11-19
BRL rate: 5.892


In [42]:
history_url = (
    "https://api.frankfurter.dev/v1/"
    "2020-01-06..2020-01-19"
)

r = requests.get(
    history_url,
    params=params,
    timeout=10
)

r.raise_for_status()
history = r.json()

print("Days of rates:", len(history["rates"]))
print(
    "BRL on 2020-01-15:",
    history["rates"]["2020-01-15"]["BRL"]
)

Days of rates: 10
BRL on 2020-01-15: 4.1599


**Your answer:**

The latest exchange rate I got was 5.892 BRL for 1 USD on November 19, 2026. For the historical data, the API returned 10 days of rates between January 6 and Janurary 19, 2020. The BRL rate on January 15, 2020 was 4.1599.

There are only 10 days of rates instead of 14 because the exchange rates are not published on weekends. The API only includes the working days in that period.

## Task 10

Frankfurter's two options are `base` and `symbols` (Task 9). Run these three commands, paste each reply, and say in one sentence how the API treated the options in each. Then say whether checking the status code alone would have caught any of the problems.

```bash
curl -s "https://api.frankfurter.dev/v1/latest?Base=USD&symbols=BRL"
curl -s "https://api.frankfurter.dev/v1/latest?base=USD&symbol=BRL"
curl -s "https://api.frankfurter.dev/v1/latest?base=usd&symbols=brl"
```


In [43]:
%%bash

curl -s "https://api.frankfurter.dev/v1/latest?Base=USD&symbols=BRL"

{"amount":1.0,"base":"EUR","date":"2026-11-19","rates":{"BRL":6.4765}}

In [44]:
%%bash

curl -s "https://api.frankfurter.dev/v1/latest?base=USD&symbol=BRL"

{"amount":1.0,"base":"USD","date":"2026-11-19","rates":{"AUD":1.49,"BRL":5.892,"CAD":1.43,"CHF":0.84261,"CNY":6.6667,"CZK":22.223,"DKK":6.7997,"EUR":0.90975,"GBP":0.76038,"HKD":7.8456,"HUF":336.09,"IDR":18261,"ILS":3.2887,"INR":93.3,"ISK":124.82,"JPY":159.97,"KRW":1328.45,"MXN":18.9183,"MYR":4.1125,"NOK":9.6866,"NZD":1.8095,"PHP":63.695,"PLN":4.0184,"RON":4.8647,"SEK":10.141,"SGD":1.2897,"THB":33.905,"TRY":50.468,"ZAR":17.2806}}

In [45]:
%%bash

curl -s "https://api.frankfurter.dev/v1/latest?base=usd&symbols=brl"

{"amount":1.0,"base":"USD","date":"2026-11-19","rates":{"BRL":5.892}}

**Your answer:**

The first request used Base instead of base, so the API ignored that parameter and used EUR as the default currency instead of USD. It still returned a result, but it was not the exchange rate I wanted.

The second request used symbol instead of symbols. The API ignored that parameter too, so it returned rates for many currencies instead of only BRL.

The third request used lowercase currency codes (usd and brl), but the API still accepted them and returned the correct USD to BRL rate of 5.892.

The first two requests show why we cannot just check whether an API returns a successful response. The request might run without an error but still give us the wrong data because of a mistake in the query parameters.


## Bonus task

Write a `while` loop that reads `data/wb_gdppc_2023_page{n}.json`, starting at `n = 1`, and extends a list called `records`, which starts empty. Stop using the `pages` field from the metadata. Do not hard-code 3. Print `len(records)` at the end: it should match the `total` field, which is 265.

In [ ]:
# Bonus task

When you are done, run all the cells from the top one more time and save the notebook. Then stage, commit and push everything to your fork: `quiz04.ipynb`, `montevideo.json`, your edited `scripts/pull_data.py` and the two files in `data/raw/`. Post your fork's link on Canvas.

Best of luck!

*For task 07, I was trying to run it in the code chunk and it was not working when I typed in $BASE$Q so i asked ChatGPT about this and sorted it out. (could have solved this on my own: I missed the ?) Also it was used to help me explain some API responses. I ran all the commands myself and used my own outputs for the tasks.*